In [41]:
%load_ext autoreload
%autoreload 2

In [42]:
import pandas as pd
import numpy as np
from pyMyriad import *
from pyMyriad.tabular import flatten, tabulate
from pyMyriad.listing import gt_table, simple_table, cascade_table

In [43]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "Education": np.random.normal(0, 1, 1000),
  "Employed": np.random.choice([0, 1], 1000)
})

In [44]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
efun = lambda df: np.mean(df.Education)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age Group", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun, label = "Income analysis")\
  .analyze_by(m = efun, label = "Education analysis")

res = atree.run(df)


# test

In [47]:
cascade_table(
    res,
    by="df.Gender",
    pivot_statistics=True
)

,_Level_0,_Level_1,_Level_2,_Level_3,F||m,F||n,M||m,M||n
0,Benin Y/N,False,Age Group,age40,-0.054519,NaN,-0.026593,NaN
1,,,,,50493.062383,13650.855255,51609.237273,14757.656895
2,,,,age60,0.128818,NaN,0.004561,NaN
3,,,,,52407.457902,13358.809054,50960.15353,13494.916687
4,,True,,age40,0.04603,NaN,0.024205,NaN
5,,,,,49791.24294,15067.170182,50735.94419,16006.577277
6,,,,age60,-0.25943,NaN,0.012993,NaN
7,,,,,50661.26192,14995.006073,50072.764027,15609.11454


# ---------------------------
# Tabulation with simple_table()

The `simple_table()` function provides a lightweight way to display analysis results as a pandas DataFrame without requiring the great-tables package.

In [23]:
# Basic simple_table - shows only analysis results
simple_table(res)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
7,Gender,F,Benin Y/N,False,Age Group,age40,m,49917.098551
7,,,,,,,n,13967.138602
8,,,,,,,m,-0.00754
10,,,,,,age60,m,49749.37496
10,,,,,,,n,14877.115755
11,,,,,,,m,-0.070265
15,,,,True,,age40,m,52139.979864
15,,,,,,,n,14911.430629
16,,,,,,,m,-0.030544
18,,,,,,age60,m,51590.476119


## simple_table with pivot

You can pivot by a split variable to show results across columns:

In [24]:
# Pivot by Gender to show results side-by-side
simple_table(res, by="df.Gender")

pivot_lvl,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,F,M
0,Benin Y/N,False,Age Group,age40,m,-0.00754,0.09266
1,,,,,n,NaN,NaN
2,,,,,m,49917.098551,51718.347075
3,,,,,n,13967.138602,14540.961617
4,,,,age60,m,-0.070265,0.073461
5,,,,,n,NaN,NaN
6,,,,,m,49749.37496,51260.364007
7,,,,,n,14877.115755,9711.593984
8,,True,,age40,m,-0.030544,0.058588
9,,,,,n,NaN,NaN


## cascade_table - show all tree nodes

Include all tree nodes (splits, summaries, and analyses) using `cascade_table()`:

In [25]:
# Show all nodes including splits
cascade_table(res).head(10)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
0,--,--,--,--,--,--,None,None
1,Gender,--,--,--,--,--,None,None
2,,F,--,--,--,--,None,None
3,,,Benin Y/N,--,--,--,None,None
4,,,,False,--,--,None,None
5,,,,,Age Group,--,None,None
6,,,,,,age40,None,None
7,,,,,,,m,49917.098551
7,,,,,,,n,13967.138602
8,,,,,,,m,-0.00754


## simple_table without duplicate suppression

By default, duplicate values in hierarchy columns are suppressed for cleaner display. You can disable this:

In [26]:
# Show all duplicate values
simple_table(res, suppress_duplicates=False).head()

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
7,Gender,F,Benin Y/N,False,Age Group,age40,m,49917.098551
7,Gender,F,Benin Y/N,False,Age Group,age40,n,13967.138602
8,Gender,F,Benin Y/N,False,Age Group,age40,m,-0.00754
10,Gender,F,Benin Y/N,False,Age Group,age60,m,49749.37496
10,Gender,F,Benin Y/N,False,Age Group,age60,n,14877.115755


# Tabulation with gt_table()

The `gt_table()` function creates beautifully formatted tables using the great-tables package. It provides more styling options and is ideal for reports and presentations.

In [27]:
# Basic gt_table with title
gt_table(res, title="Analysis Results").show()

Analysis Results 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 49917.09855146416 
 
 
 
 
 
 
 
 
 n 
 13967.138601906134 
 
 
 
 
 
 
 
 
 m 
 -0.007539970677157397 
 
 
 
 
 
 
 
 age60 
 m 
 49749.37496032763 
 
 
 
 
 
 
 
 
 n 
 14877.115755061024 
 
 
 
 
 
 
 
 
 m 
 -0.07026486556807725 
 
 
 
 
 
 True 
 
 age40 
 m 
 52139.97986403375 
 
 
 
 
 
 
 
 
 n 
 14911.430628705813 
 
 
 
 
 
 
 
 
 m 
 -0.030544437122748798 
 
 
 
 
 
 
 
 age60 
 m 
 51590.47611920429 
 
 
 
 
 
 
 
 
 n 
 11907.600786077222 
 
 
 
 
 
 
 
 
 m 
 0.15812391991428978 
 
 
 
 M 
 
 False 
 
 age40 
 m 
 51718.34707454203 
 
 
 
 
 
 
 
 
 n 
 14540.96161742721 
 
 
 
 
 
 
 
 
 m 
 0.09265964189195633 
 
 
 
 
 
 
 
 age60 
 m 
 51260.36400720145 
 
 
 
 
 
 
 
 
 n 
 9711.593984225141 
 
 
 
 
 
 
 
 
 m 
 0.07346117773606944 
 
 
 
 
 
 True 
 
 age40 
 m 
 49285.02307157711 
 
 
 
 
 
 
 
 
 n 
 15098.878349576633 
 
 
 
 
 
 
 
 
 m 
 0.058588139495610876 
 
 
 
 
 
 
 
 age60 
 m 
 47353.323322932105 
 
 
 
 
 
 
 
 
 n 
 14369.690582715783 
 
 
 
 
 
 
 
 
 m 
 0.2044744216819571

## gt_table with pivot

Pivot by a split variable:

In [28]:
# Pivot by Gender with custom title and subtitle
gt_table(
    res, 
    by="df.Gender", 
    title="Gender Comparison",
    subtitle="Income and Education Analysis by Gender"
).show()

Gender Comparison 
 
 
 Income and Education Analysis by Gender 
 
 
 
 Hierarchy 
 
 Statistic 
 F 
 M 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 -0.007539970677157397 
 0.09265964189195633 
 
 
 
 
 
 
 n 
 
 
 
 
 
 
 
 
 m 
 49917.09855146416 
 51718.34707454203 
 
 
 
 
 
 
 n 
 13967.138601906134 
 14540.96161742721 
 
 
 
 
 
 age60 
 m 
 -0.07026486556807725 
 0.07346117773606944 
 
 
 
 
 
 
 n 
 
 
 
 
 
 
 
 
 m 
 49749.37496032763 
 51260.36400720145 
 
 
 
 
 
 
 n 
 14877.115755061024 
 9711.593984225141 
 
 
 
 True 
 
 age40 
 m 
 -0.030544437122748798 
 0.058588139495610876 
 
 
 
 
 
 
 n 
 
 
 
 
 
 
 
 
 m 
 52139.97986403375 
 49285.02307157711 
 
 
 
 
 
 
 n 
 14911.430628705813 
 15098.878349576633 
 
 
 
 
 
 age60 
 m 
 0.15812391991428978 
 0.2044744216819571 
 
 
 
 
 
 
 n 
 
 
 
 
 
 
 
 
 m 
 51590.47611920429 
 47353.323322932105 
 
 
 
 
 
 
 n 
 11907.600786077222 
 14369.690582715783

## gt_table with all nodes and custom formatting

Include non-analysis nodes and customize decimal places:

In [29]:
# Show all nodes with 2 decimal places
gt_table(
    res, 
    title="Complete Analysis Tree",
    cascade=True,
    decimals=2
).show()

Complete Analysis Tree 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 Gender 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 F 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49917.09855146416 
 
 
 
 
 
 
 
 
 n 
 13967.138601906134 
 
 
 
 
 
 
 
 
 m 
 -0.007539970677157397 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 49749.37496032763 
 
 
 
 
 
 
 
 
 n 
 14877.115755061024 
 
 
 
 
 
 
 
 
 m 
 -0.07026486556807725 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 52139.97986403375 
 
 
 
 
 
 
 
 
 n 
 14911.430628705813 
 
 
 
 
 
 
 
 
 m 
 -0.030544437122748798 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 51590.47611920429 
 
 
 
 
 
 
 
 
 n 
 11907.600786077222 
 
 
 
 
 
 
 
 
 m 
 0.15812391991428978 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 51718.34707454203 
 
 
 
 
 
 
 
 
 n 
 14540.96161742721 
 
 
 
 
 
 
 
 
 m 
 0.09265964189195633 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 51260.36400720145 
 
 
 
 
 
 
 
 
 n 
 9711.593984225141 
 
 
 
 
 
 
 
 
 m 
 0.07346117773606944 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49285.02307157711 
 
 
 
 
 
 
 
 
 n 
 15098.878349576633 
 
 
 
 
 
 
 
 
 m 
 0.058588139495610876 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 47353.323322932105 
 
 
 
 
 
 
 
 
 n 
 14369.690582715783 
 
 
 
 
 
 
 
 
 m 
 0.2044744216819571

# Pivoting Statistics as Columns

The `pivot_statistics` parameter allows you to display statistics as columns instead of rows, creating a wider, more compact table format.

In [30]:
# Pivot statistics into columns - simple_table
simple_table(res, pivot_statistics=True)

statistics,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,m,n
0,Gender,F,Benin Y/N,False,Age Group,age40,-0.00754,NaN
1,,,,,,,49917.098551,13967.138602
2,,,,,,age60,-0.070265,NaN
3,,,,,,,49749.37496,14877.115755
4,,,,True,,age40,-0.030544,NaN
5,,,,,,,52139.979864,14911.430629
6,,,,,,age60,0.158124,NaN
7,,,,,,,51590.476119,11907.600786
8,,M,,False,,age40,0.09266,NaN
9,,,,,,,51718.347075,14540.961617


## Combining pivot_statistics with by

You can combine both pivots to create a matrix-style table with split groups and statistics as columns:

In [31]:
# Combine both pivots - shows Gender groups with statistics as columns
simple_table(res, by="df.Gender", pivot_statistics=True)

,_Level_0,_Level_1,_Level_2,_Level_3,F||m,F||n,M||m,M||n
0,Benin Y/N,False,Age Group,age40,-0.00754,NaN,0.09266,NaN
1,,,,,49917.098551,13967.138602,51718.347075,14540.961617
2,,,,age60,-0.070265,NaN,0.073461,NaN
3,,,,,49749.37496,14877.115755,51260.364007,9711.593984
4,,True,,age40,-0.030544,NaN,0.058588,NaN
5,,,,,52139.979864,14911.430629,49285.023072,15098.87835
6,,,,age60,0.158124,NaN,0.204474,NaN
7,,,,,51590.476119,11907.600786,47353.323323,14369.690583


## gt_table with pivot_statistics

The same feature works with `gt_table()` for beautifully formatted output:

In [33]:
# gt_table with statistics as columns
gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    title="Analysis with Statistics as Columns"
).show()

Analysis with Statistics as Columns 
 
 
 
 Hierarchy 
 
 
 F 
 
 
 M 
 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 m 
 n 
 m 
 n 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 -0.007539970677157397 
 
 0.09265964189195633 
 
 
 
 
 
 
 
 49917.09855146416 
 13967.138601906134 
 51718.34707454203 
 14540.96161742721 
 
 
 
 
 
 age60 
 -0.07026486556807725 
 
 0.07346117773606944 
 
 
 
 
 
 
 
 49749.37496032763 
 14877.115755061024 
 51260.36400720145 
 9711.593984225141 
 
 
 
 True 
 
 age40 
 -0.030544437122748798 
 
 0.058588139495610876 
 
 
 
 
 
 
 
 52139.97986403375 
 14911.430628705813 
 49285.02307157711 
 15098.878349576633 
 
 
 
 
 
 age60 
 0.15812391991428978 
 
 0.2044744216819571 
 
 
 
 
 
 
 
 51590.47611920429 
 11907.600786077222 
 47353.323322932105 
 14369.690582715783

In [34]:
# gt_table combining both pivots
gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    title="Gender Comparison - Matrix View",
    subtitle="Each group-statistic combination as a column"
).show()

Gender Comparison - Matrix View 
 
 
 Each group-statistic combination as a column 
 
 
 
 Hierarchy 
 
 
 F 
 
 
 M 
 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 m 
 n 
 m 
 n 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 -0.007539970677157397 
 
 0.09265964189195633 
 
 
 
 
 
 
 
 49917.09855146416 
 13967.138601906134 
 51718.34707454203 
 14540.96161742721 
 
 
 
 
 
 age60 
 -0.07026486556807725 
 
 0.07346117773606944 
 
 
 
 
 
 
 
 49749.37496032763 
 14877.115755061024 
 51260.36400720145 
 9711.593984225141 
 
 
 
 True 
 
 age40 
 -0.030544437122748798 
 
 0.058588139495610876 
 
 
 
 
 
 
 
 52139.97986403375 
 14911.430628705813 
 49285.02307157711 
 15098.878349576633 
 
 
 
 
 
 age60 
 0.15812391991428978 
 
 0.2044744216819571 
 
 
 
 
 
 
 
 51590.47611920429 
 11907.600786077222 
 47353.323322932105 
 14369.690582715783

In [36]:
from pyMyriad.tabular import format_statistics

# format the 'Income analysis' analysis (identified with its label) into a new statistic 'm' with a custom format, and remove the original statistics
rr = format_statistics(
    res, 
    label="Income analysis",  # Only format nodes with this label
    m="{m:.0f} /// {n:.0f}",         # Create statistic 'm' with this format
    inplace=False, 
    remove_original=True
)

gt_table(rr, title="Formatted Statistics").show()

Formatted Statistics 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 49917 /// 13967 
 
 
 
 
 
 
 
 
 m 
 -0.007539970677157397 
 
 
 
 
 
 
 
 age60 
 m 
 49749 /// 14877 
 
 
 
 
 
 
 
 
 m 
 -0.07026486556807725 
 
 
 
 
 
 True 
 
 age40 
 m 
 52140 /// 14911 
 
 
 
 
 
 
 
 
 m 
 -0.030544437122748798 
 
 
 
 
 
 
 
 age60 
 m 
 51590 /// 11908 
 
 
 
 
 
 
 
 
 m 
 0.15812391991428978 
 
 
 
 M 
 
 False 
 
 age40 
 m 
 51718 /// 14541 
 
 
 
 
 
 
 
 
 m 
 0.09265964189195633 
 
 
 
 
 
 
 
 age60 
 m 
 51260 /// 9712 
 
 
 
 
 
 
 
 
 m 
 0.07346117773606944 
 
 
 
 
 
 True 
 
 age40 
 m 
 49285 /// 15099 
 
 
 
 
 
 
 
 
 m 
 0.058588139495610876 
 
 
 
 
 
 
 
 age60 
 m 
 47353 /// 14370 
 
 
 
 
 
 
 
 
 m 
 0.2044744216819571